# 9장 — 조련 성적표: 평가와 부작용 📊

웹 챕터의 오차범위 실험을 뒤집어서 해 봅니다 — **같은 시험을 수백 번 보게 하는** 시뮬레이션.

1. 진짜 실력 72% vs 66%인 두 모델이 **n문항 시험에서 순위가 뒤집히는 빈도**를 세고
2. 진짜 실력이 **똑같은** 두 모델이 우연히 몇 %p 차이로 보이는지도 봅니다

3부 8~9장(신뢰구간·가설검정)이 왜 필요한지, 숫자가 직접 말해 줄 거예요.

In [ ]:
# 🤖 AI 튜터 — 공부하다 막히면 오른쪽 아래 버튼 (로컬 Ollama, 없어도 학습 지장 없음)
from tutor import chat
chat('ch9')

## 1. 시험 시뮬레이터

모델의 "진짜 실력" = 문항을 맞힐 확률 p. n문항 시험 점수는 **이항분포**(3부 5장)에서 뽑힌
표본 평균입니다 — 같은 모델이라도 시험마다 점수가 출렁여요.

In [ ]:
import numpy as np

rng = np.random.default_rng(42)

def take_exam(true_skill, n_items, n_trials):
    """진짜 실력 true_skill 인 모델이 n문항 시험을 n_trials번 봤을 때의 점수들(%)."""
    return rng.binomial(n_items, true_skill, size=n_trials) / n_items * 100

# 맛보기 — 진짜 실력 72%짜리 모델이 100문항 시험을 5번 보면?
print('진짜 실력 72% 모델의 100문항 시험 점수 5회:', take_exam(0.72, 100, 5))

## 2. 실험 A — 순위가 뒤집히는 빈도

진짜 실력: A = 72%, B = 66% (A가 확실히 낫다).
그런데 n문항 시험 **한 번**으로 순위를 매기면, B가 이기는 사고가 얼마나 자주 날까요?

In [ ]:
TRIALS = 2000

def flip_rate(n_items):
    a = take_exam(0.72, n_items, TRIALS)
    b = take_exam(0.66, n_items, TRIALS)
    return float((b >= a).mean())

for n in (50, 100, 400, 2000):
    rate = flip_rate(n)
    bar = '█' * int(rate * 100)
    print(f'n={n:5d}문항: B가 이기거나 비기는 확률 {rate:5.1%} {bar}')

6%p나 차이 나는 두 모델인데 — 문항이 적으면 **열 번에 한두 번은 순위가 뒤집힙니다**.
리더보드 스냅숏 한 장을 믿으면 안 되는 이유예요.

## 3. 실험 B — 실력이 같아도 차이는 난다

이번엔 둘 다 진짜 실력 70%. 그런데 100문항 시험 한 번으로 비교하면?

In [ ]:
a = take_exam(0.70, 100, TRIALS)
b = take_exam(0.70, 100, TRIALS)
gap = a - b
fake_win_3pp = float((np.abs(gap) >= 3).mean())
print(f'실력이 똑같은데 |점수 차| ≥ 3%p 로 보일 확률: {fake_win_3pp:.1%}')
print(f'관찰된 최대 가짜 격차: {np.abs(gap).max():.0f}%p')

In [ ]:
import matplotlib.pyplot as plt

gap_small = take_exam(0.70, 100, TRIALS) - take_exam(0.70, 100, TRIALS)
gap_large = take_exam(0.70, 2000, TRIALS) - take_exam(0.70, 2000, TRIALS)

plt.figure(figsize=(6.5, 3.2))
plt.hist(gap_small, bins=30, alpha=0.6, color='#f59e0b', label='n=100 (가짜 격차가 넓게 퍼짐)')
plt.hist(gap_large, bins=30, alpha=0.7, color='#6366f1', label='n=2000 (0 근처로 수렴)')
plt.xlabel('두 모델의 점수 차 (%p) — 진짜 실력은 동일!')
plt.ylabel('빈도')
plt.title('실력이 같아도 시험은 차이를 만든다 — 문항 수가 그 폭을 정한다')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. 검증

In [ ]:
rate_100, rate_2000 = flip_rate(100), flip_rate(2000)

# 검증 1 — 문항이 적을수록 순위 뒤집힘이 잦다
assert rate_100 > rate_2000, '문항이 적을수록 순위 사고가 잦아야 한다!'
# 검증 2 — 문항이 충분하면 순위는 사실상 안정된다
assert rate_2000 < 0.005, 'n=2000이면 6%p 차이는 사실상 항상 드러나야 한다!'
# 검증 3 — 실력이 같아도 n=100에선 3%p 이상 가짜 격차가 드물지 않다
assert fake_win_3pp > 0.3, '실력이 같아도 작은 시험은 가짜 격차를 자주 만든다!'
print(f'✔ 확인: 뒤집힘 {rate_100:.1%}(n=100) → {rate_2000:.2%}(n=2000) / 가짜 3%p 격차 {fake_win_3pp:.0%}')
print('✔ 점수 차이를 읽기 전에 n 을 보라 — 3부 8~9장이 실무가 되는 순간')

## 5. 관찰 정리 + 🤖 AI 연결

- 벤치마크 점수 = 이항분포에서 뽑힌 **표본 평균** — 시험마다 출렁이고, 폭은 1/√n.
- 진짜 실력이 6%p 차이 나도 작은 시험(n=100)에선 열에 한 번꼴로 순위가 뒤집히고,
  실력이 **같아도** 3%p 이상 가짜 격차가 흔하다(약 3분의 1) — "1.2%p 앞섰다"는 홍보의 실체.
- **회수한 수학**: 3부 5장(이항분포·표본), 8장(신뢰구간 — ±의 정체), 9장(가설검정 —
  우연인가 진짜인가). 심화 트랙이 평가 문해력이라는 실전 기술이 됐습니다.
- 아첨·망각·거절 과잉 같은 부작용은 점수에 안 잡힙니다 — 전용 평가셋으로 따로 감시(웹 챕터 진단소).

**다음 챕터**: 10장 — 🎯 캡스톤. SFT → 보상 모델 → DPO, 조련 파이프라인 전체를 조립하며
시리즈의 여정을 마무리합니다.

> 💡 궁금한 코드는 그 셀을 실행한 다음 셀에서 `from tutor import explain` 후 `explain('ch9')`.